# Math 638 Project 2 - Homework 5.2

## Next-Token Prediction for the Lorenz Dynamical System

This report presents Homework 5.2 in the same **formal deliverables style** used for Homework 5.1. Because Homework 5.2 does not provide a separate deliverables list, the numbered tasks are converted into a reproducible set of report deliverables.

The project studies the classical Lorenz system

$$
\dot{x}=\sigma(y-x),\qquad
\dot{y}=x(\rho-z)-y,\qquad
\dot{z}=xy-\beta z,
$$

with

$$
\sigma=10,\qquad \rho=28,\qquad \beta=\frac{8}{3}.
$$

The central question is:

**How much of the temporal structure of a continuous chaotic system can be represented and generated by a discrete next-token model?**

The workflow is:

**Lorenz trajectories -> phase-space boxes -> token books -> Markov next-token model -> generated token trajectories.**

All numerical results reported below come from the corrected saved outputs in `data/hw5_2/`. Figures are referenced with portable relative Markdown paths so the notebook can be exported through VS Code/Jupyter without specialized LaTeX document tooling.

## Executive Summary

The Lorenz system is a useful sequence-modeling example because it is low-dimensional and deterministic, yet chaotic. For the classical parameters used here,

$$
\nabla\cdot F
=
-\sigma-1-\beta
=
-\frac{41}{3}<0,
$$

so the system is dissipative.

The experiment generates 60 independent Lorenz trajectories: 40 training books, 10 validation books, and 10 test books. Each trajectory discards a 5-time-unit transient and retains 25 time units sampled every

$$
\Delta t=0.02,
$$

giving 1,251 samples per book.

The baseline discretization divides each coordinate into 12 bins, creating

$$
12^3=1728
$$

possible tokens. Only 250 boxes occur in the training data and 257 occur across all saved trajectories.

Empirical next-token models with context lengths

$$
m=1,2,3
$$

are compared. Validation selects

$$
m=3.
$$

Final test accuracy increases from 0.60696 at $m=1$ to 0.75913 at $m=3$, while exact length-3 history coverage remains 0.99447. Thus the improvement is not mainly caused by fallback to shorter histories.

The selected model is then used to generate 100 stochastic token trajectories. A representation-matched Task 8 diagnostic shows that generated box centers occupy essentially the same **coarse tokenized support** as held-out true box centers. The more informative lobe-switching statistics are also close: true test box-center trajectories average 11.52 switches per 1000 steps and generated trajectories average 11.77, while mean residence times are about 1.67 and 1.65 time units.

Finally, changing the discretization from 8 to 20 bins per coordinate reduces test quantization RMSE from 3.06328 to 1.22666, but test next-token accuracy decreases from 0.77604 to 0.73886 and unseen exact histories become more common.

The conclusion is deliberately limited: the discrete Markov generator reproduces selected **coarse sequence statistics** of the Lorenz system. It does not establish equality of invariant measures, Lyapunov exponents, entropy, or exact continuous-time Lorenz dynamics.

## Deliverables Overview

| Deliverable | Content | Status |
| --- | --- | --- |
| 1 | Chosen dynamical system and mathematical classification | Complete |
| 2 | Independent trajectory books and train/validation/test construction | Complete |
| 3 | Phase-space discretization and vocabulary definition | Complete |
| 4 | Fixed-time tokenization and data checks | Complete |
| 5 | First-order and higher-order next-token model | Complete |
| 6 | Context-length comparison and validation-based selection | Complete |
| 7 | Generated token trajectories and sampling protocol | Complete |
| 8 | Lorenz attractor-support and two-lobe switching diagnostics | Complete |
| 9 | Hamiltonian invariant applicability | Not applicable to dissipative Lorenz choice |
| 10 | Coarse/fine discretization sensitivity | Complete |
| 11 | Reproducibility and submitted-output record | Complete |
| 12 | LLM critic prompt and substantive response | Complete |
| 13 | Human critique, limitations, and interpretation boundaries | Complete |

# 1. Dynamical System Choice and Mathematical Classification

For this project, the selected system is the Lorenz system

$$
\dot{x}=\sigma(y-x),
$$

$$
\dot{y}=x(\rho-z)-y,
$$

$$
\dot{z}=xy-\beta z,
$$

with

$$
\sigma=10,\qquad \rho=28,\qquad \beta=\frac{8}{3}.
$$

The vector field is

$$
F(x,y,z)
=
(
\sigma(y-x),
x(\rho-z)-y,
xy-\beta z
).
$$

Its divergence is

$$
\nabla\cdot F
=
\frac{\partial \dot{x}}{\partial x}
+
\frac{\partial \dot{y}}{\partial y}
+
\frac{\partial \dot{z}}{\partial z}
=
-\sigma-1-\beta.
$$

Therefore

$$
\nabla\cdot F
=
-10-1-\frac{8}{3}
=
-\frac{41}{3}<0.
$$

Hence the Lorenz system is **dissipative**: infinitesimal phase-space volumes contract under the flow.

For the classical parameters used here, the system is also **chaotic** and exhibits sensitive dependence on initial conditions together with the familiar two-lobed Lorenz attractor.

The system is deterministic because the ODE and initial condition determine the trajectory. It is not treated as Hamiltonian in this experiment, and in this classical chaotic regime it is not treated as an integrable system reducible to a sufficient family of conserved quantities.

The Lorenz system is a good choice for Homework 5.2 because it combines simple low-dimensional equations with nontrivial temporal structure. This makes it possible to test whether a discrete next-token model can learn meaningful sequence regularities without confusing predictability with exact recovery of the continuous dynamics.

# 2. Trajectory Collection and "Book" Construction

The experiment generates independent trajectories from different seeded initial conditions.

The fixed design is:

- random seed: `63852`;
- 40 training trajectories;
- 10 validation trajectories;
- 10 test trajectories;
- initial-condition region

$$
x_0,y_0\in[-20,20],
\qquad
z_0\in[5,35];
$$

- 5 time units discarded as transient;
- 25 retained time units;
- sampling interval

$$
\Delta t=0.02;
$$

- 1,251 retained samples per trajectory.

The trajectories are integrated numerically with DOP853 using fixed tight solver tolerances.

Each trajectory is treated as one independent training "book." This is important because a transition from the final state of one initial condition to the first state of another would be artificial and has no dynamical meaning.

| Split | Books | Samples per book | Total retained samples |
| --- | ---: | ---: | ---: |
| Training | 40 | 1,251 | 50,040 |
| Validation | 10 | 1,251 | 12,510 |
| Test | 10 | 1,251 | 12,510 |

The validation and test books remain distinct from the training books throughout model fitting.

![Example Lorenz trajectory](../result%20images/hw5_2/step2_example_lorenz_3d.png)

![Five independent Lorenz training books](../result%20images/hw5_2/step2_five_training_books_xz.png)

# 3. Phase-Space Discretization and Vocabulary

The continuous state

$$
X(t)=(x(t),y(t),z(t))
$$

must be converted into a finite token vocabulary.

The baseline discretization uses

$$
N_x=N_y=N_z=12
$$

uniform bins. Hence the possible vocabulary size is

$$
|\mathcal V_{\mathrm{possible}}|
=
N_xN_yN_z
=
12^3
=
1728.
$$

The phase-space bounds are fitted using the **training trajectories only** and then padded by 5%. The frozen bounds are approximately

$$
x\in[-20.5511,20.7604],
$$

$$
y\in[-27.9241,28.6170],
$$

$$
z\in[0.6595,48.7944].
$$

Validation and test data do not determine these bounds.

Only a subset of the possible boxes is actually visited:

- possible vocabulary: 1,728 boxes;
- observed training vocabulary: 250 boxes;
- observed vocabulary across all saved trajectories: 257 boxes.

The saved run has zero clipped samples in training, validation, and test, so every retained trajectory point lies inside the padded training-fitted region.

The distinction between **possible vocabulary** and **observed vocabulary** is important. The possible vocabulary grows cubically with the number of bins per axis,

$$
|\mathcal V_{\mathrm{possible}}|=N^3,
$$

while the observed vocabulary depends on which parts of phase space are actually visited.

# 4. Token Sequence Construction and Data Checks

Let the bin indices of a sampled state be

$$
(i_x,i_y,i_z).
$$

For the baseline grid, each box is assigned one integer token according to

$$
a
=
i_x(N_yN_z)
+
i_yN_z
+
i_z.
$$

Thus the sampled trajectory

$$
X(t_1),X(t_2),\ldots,X(t_L)
$$

becomes the discrete sequence

$$
a_1,a_2,\ldots,a_L.
$$

Every trajectory is tokenized separately. No context or transition is allowed to cross a book boundary.

The same sampling interval and same training-fitted discretization are used for all books.

The resulting token books are stored in:

- `data/hw5_2/token_books/train/`
- `data/hw5_2/token_books/validation/`
- `data/hw5_2/token_books/test/`

The complete baseline token-to-box geometry is stored in:

- `data/hw5_2/baseline_box_vocabulary.csv`

The token-frequency distribution is highly nonuniform because the Lorenz attractor occupies only a small subset of the rectangular grid.

![Thirty most frequent baseline tokens](../result%20images/hw5_2/step4_top30_token_frequencies.png)

# 5. Next-Token Model and Estimation Procedure

## 5.1 First-order model

For a first-order empirical Markov model, let $C(a,b)$ be the number of training transitions from token $a$ to token $b$, and let $C(a)$ be the number of training occurrences of $a$ as a current token. Then

$$
P(a_{n+1}=b\mid a_n=a)=\frac{C(a,b)}{C(a)}.
$$

Counts are accumulated separately inside each training book.

## 5.2 Higher-order model

For a context length $m$, let $C(a_{n-m+1},\ldots,a_n,b)$ be the number of training occurrences of the context followed by $b$, and let $C(a_{n-m+1},\ldots,a_n)$ be the number of training occurrences of the context. Then

$$
P(a_{n+1}=b\mid a_{n-m+1},\ldots,a_n)
=
\frac{C(a_{n-m+1},\ldots,a_n,b)}{C(a_{n-m+1},\ldots,a_n)}.
$$

The tested values are

$$
m\in\{1,2,3\}.
$$

## 5.3 Backoff rule

If an exact history is unseen in training, the predictor uses deterministic backoff:

1. try the full length-$m$ history;
2. drop the oldest token and try a shorter suffix;
3. continue until a seen history is found;
4. if no history is available, use the most frequent training token.

Exact-history coverage is always reported separately, so fallback cannot be hidden inside the accuracy statistic.

## 5.4 Example empirical first-order transition distribution

| Current token | Next token | Transition count | Empirical probability |
| --- | --- | --- | --- |
| 940 | 940 | 1358 | 0.73485 |
| 940 | 939 | 352 | 0.19048 |
| 940 | 952 | 46 | 0.02489 |
| 940 | 1084 | 34 | 0.01840 |
| 940 | 1096 | 29 | 0.01569 |
| 940 | 951 | 15 | 0.00812 |
| 940 | 1095 | 14 | 0.00758 |

# 6. Context-Length Comparison and Model Selection

The model-selection protocol uses **validation only**. The test split is not used to choose the context length.

| m | Validation accuracy | Test accuracy | Test exact-history coverage | Test unseen exact-history fraction | Selected by validation |
| --- | --- | --- | --- | --- | --- |
| 1 | 0.61448 | 0.60696 | 0.99944 | 0.00056 | No |
| 2 | 0.69952 | 0.69488 | 0.99808 | 0.00192 | No |
| 3 | 0.75721 | 0.75913 | 0.99447 | 0.00553 | Yes |

Validation selects

$$
m=3
$$

because it achieves the highest validation accuracy among the predeclared candidates.

The corresponding final test accuracy is

$$
0.75913,
$$

compared with

$$
0.60696
$$

for $m=1$.

At $m=3$, exact-history coverage on the test set is

$$
0.99447,
$$

so the accuracy gain is not mainly explained by fallback to shorter contexts.

The supported conclusion is limited to the tested range:

> More context improved prediction from $m=1$ through $m=3$ under the fixed baseline discretization and sampling interval.

The experiment does **not** establish that $m=3$ is globally optimal or that still longer contexts would not help.

![Prediction accuracy versus context length](../result%20images/hw5_2/step6_accuracy_vs_context_length.png)

![Unseen exact histories versus context length](../result%20images/hw5_2/step6_unseen_history_vs_context_length.png)


# 7. Generated Token Trajectories

The validation-selected $m=3$ model is fitted using the training token books.

For each held-out test book, only the first three tokens are used as a seed context. Future test tokens are not supplied to the generator.

Subsequent tokens are sampled autoregressively:

$$
a_{n+1}
\sim
P(
\cdot
\mid
a_{n-2},a_{n-1},a_n
).
$$

Ten stochastic replicates are generated from each of the 10 test seeds, producing:

- 100 generated trajectories;
- 1,251 tokens per generated trajectory;
- 124,800 generated prediction steps after the seed contexts.

Of these prediction steps:

- 124,787 use the full selected length-3 context;
- only 13 use shorter-history backoff.

Thus essentially all generated transitions are sampled from the full selected context model.

Generated tokens are converted back to approximate continuous coordinates using the centers of their phase-space boxes:

$$
a_i
\longrightarrow
(\hat x_i,\hat y_i,\hat z_i).
$$

These box-center trajectories are discrete reconstructions, not numerical solutions of the Lorenz ODE.

![True and generated Lorenz trajectories](../result%20images/hw5_2/step8_true_vs_generated_xz.png)

# 8. Lorenz Attractor Support and Two-Lobe Switching

## 8.1 Representation-matched attractor-support diagnostic


Because a generated token reconstructs only to a box center, the primary geometric comparison must use the **same representation** for held-out truth and generated data.

Let

$$
\mathcal A_{\mathrm{train}}
$$

be the continuous training-state point cloud. For a box-center state $q$, define

$$
d(q)
=
\min_{p\in\mathcal A_{\mathrm{train}}}
\|q-p\|_2.
$$

Validation, test, and generated token sequences are all mapped to box centers before this distance is computed.

The threshold is fixed as the 95th percentile of validation **box-center** distances.


| Dataset | Representation | Mean distance | Median distance | p95 distance | Fraction within validation p95 |
| --- | --- | --- | --- | --- | --- |
| Validation | token box centers | 0.85727 | 0.69084 | 2.18689 | 0.95028 |
| Test | token box centers | 0.85364 | 0.65535 | 2.18689 | 0.95028 |
| Generated | token box centers | 0.84820 | 0.65535 | 2.18689 | 0.95205 |

Under this representation-matched diagnostic, generated box centers occupy essentially the same coarse support as held-out true box centers.

This is a **weak support test**. The empirical Markov model samples next tokens from training-observed transitions, so coarse support retention is partly guaranteed by construction. It should not be interpreted as proof that the generated paths are exact Lorenz trajectories.

![Attractor proximity ECDF](../result%20images/hw5_2/step8_attractor_proximity_ecdf.png)

## 8.2 Two-lobe switching

A second diagnostic uses the sign of $x$ to label the two major Lorenz lobes:

$$
x<0
\qquad
x\ge 0.
$$

For each trajectory, the experiment measures switch frequency and residence time.


| Dataset | Mean switches / 1000 steps | Median switches / 1000 steps | Mean residence time | Median residence time |
| --- | --- | --- | --- | --- |
| True test box centers | 11.520 | 11.200 | 1.6735 | 1.6680 |
| Generated box centers | 11.768 | 11.200 | 1.6531 | 1.6680 |
| True continuous test states | 11.360 | 11.200 | 1.6909 | 1.6680 |

The generated sequences reproduce these tested coarse switching summaries reasonably well.

The primary matched comparison is between **true test box centers** and **generated box centers**. The continuous true-test values are included only as a sensitivity check showing that quantization changes the switching summary only slightly.

Matching these statistics does **not** establish equality of invariant measures, Lyapunov exponents, entropy, or exact continuous-time dynamics.

![Lobe residence-time distributions](../result%20images/hw5_2/step8_lobe_residence_distribution.png)


# 9. Hamiltonian Invariant Applicability

Task 9 asks for invariant checks for a Hamiltonian or nearly Hamiltonian system.

The selected Lorenz system is dissipative:

$$
\nabla\cdot F
=
-\frac{41}{3}<0.
$$

Therefore the Hamiltonian/Kirchhoff checks are not applicable.

This report deliberately does **not** invent conservation tests for:

- Hamiltonian energy;
- linear momentum;
- angular momentum;
- Casimir invariants.

For the chosen Lorenz system, the relevant generated-sequence checks are geometric support and lobe-switching behavior.

# 10. Discretization Sensitivity

To isolate the effect of phase-space resolution, the context length is held fixed at the validation-selected value

$$
 m=3.
$$

Only the number of bins per coordinate is changed.

| Bins / axis | Possible vocabulary | Observed train vocabulary | Validation accuracy | Test accuracy | Test unseen exact-history fraction | Test quantization RMSE |
| --- | --- | --- | --- | --- | --- | --- |
| 8 | 512 | 117 | 0.77308 | 0.77604 | 0.00337 | 3.06328 |
| 12 | 1,728 | 250 | 0.75721 | 0.75913 | 0.00553 | 2.04171 |
| 20 | 8,000 | 628 | 0.74712 | 0.73886 | 0.02011 | 1.22666 |

The measured tradeoff is clear.

### Prediction

Coarser discretization produces higher next-token accuracy in the tested range:

At $8^3$ bins the test accuracy is

$$
0.77604,
$$

whereas at $20^3$ bins it is

$$
0.73886.
$$

### Geometry

Finer discretization produces smaller geometric quantization error:

At $8^3$ bins the quantization RMSE is

$$
3.06328,
$$

whereas at $20^3$ bins it is

$$
1.22666.
$$

### Sparsity

The fraction of unseen exact length-3 test histories rises from

$$
0.00337
$$

to

$$
0.02011.
$$

Thus finer boxes preserve more geometric detail but enlarge the vocabulary, increase history sparsity, and make the discrete prediction problem harder for the tested model and data volume.

This conclusion is restricted to the tested resolutions. It is not a theorem that every finer discretization must monotonically reduce accuracy.

![Prediction accuracy versus resolution](../result%20images/hw5_2/step10_accuracy_vs_resolution.png)

![Quantization error versus resolution](../result%20images/hw5_2/step10_quantization_error_vs_resolution.png)

![Vocabulary size versus resolution](../result%20images/hw5_2/step10_vocabulary_vs_resolution.png)

![Unseen histories versus resolution](../result%20images/hw5_2/step10_unseen_history_vs_resolution.png)


# 11. Reproducibility and Submitted Outputs

The complete workflow is separated into coherent notebooks.

## `codes/`

- `HW5_2_Step1_Lorenz_System.ipynb`
- `HW5_2_Steps2-4_Trajectories_Discretization_Tokens.ipynb`
- `HW5_2_Steps5-6_Markov_Context_Models.ipynb`
- `HW5_2_Steps7-8_Generation_Lorenz_Validation.ipynb`
- `HW5_2_Step10_Discretization_Sensitivity.ipynb`
- `HW5_2_Step11_LLM_Critic.ipynb`
- `HW5_2_Final_Formal_Report_DELIVERABLES_STYLE.ipynb`

## `data/hw5_2/`

Key saved outputs include:

- `trajectory_manifest.csv`
- `token_book_manifest.csv`
- `baseline_configuration.json`
- `baseline_box_vocabulary.csv`
- `task5_first_order_transition_example.csv`
- `task5_6_validation_context_results.csv`
- `task5_6_test_context_results.csv`
- `task5_6_selected_markov_model.json`
- `task7_generated_sequence_manifest.csv`
- `generated_sequences/`
- `task8_attractor_proximity.csv`
- `task8_lobe_switching_by_trajectory.csv`
- `task10_discretization_sensitivity.csv`
- `task11_fixed_evidence_summary.txt`
- `task11_llm_critic_prompt.txt`
- `task11_llm_critic_response.txt`

## `result images/hw5_2/`

The saved figures include the Lorenz trajectory visualizations, token-frequency plot, context-length comparison, attractor-support diagnostic, lobe-residence distribution, and all discretization-sensitivity plots.

## Numerical-chaos reproducibility note

The random seed, ODE solver, and numerical tolerances are fixed. However, the Lorenz system is chaotic, so tiny floating-point or software-platform differences can eventually change exact long trajectories.

Therefore the saved trajectories and derived CSV files are the authoritative realization used for this report. Reproduction should focus on the aggregate conclusions rather than requiring bitwise-identical long trajectories across every machine.

# 12. LLM Critic Prompt and Substantive Response

The numerical experiment was fixed before the LLM was asked to interpret it. The exact prompt instructions were:

```text
You are the critic for a Math 638 experiment that converts the Lorenz dynamical system into discrete token sequences and learns next-token Markov models.

The numerical experiment is already complete. Do not redesign the model and do not invent new measurements.

Use only the fixed evidence summary supplied below.

Your tasks are:
1. State which conclusions are strongly supported by the reported numbers.
2. State which conclusions are plausible but not established by the experiment.
3. Evaluate whether increasing context length actually improved next-token prediction.
4. Evaluate whether the generated sequences remain near the Lorenz attractor.
5. Evaluate whether the generated sequences reproduce the two-lobe switching behavior.
6. Explain the measured tradeoff between coarse and fine phase-space discretization.
7. Identify important observed failure modes.
8. Identify hypotheses that were NOT tested.
9. Check the physical interpretation carefully.

Important mathematical constraints:
- The chosen system is the classical Lorenz system.
- Its divergence is -41/3 < 0, so it is dissipative.
- This project did not choose the Hamiltonian/Kirchhoff system.
- Do not claim that Hamiltonian energy, momentum, or Casimir invariants were preserved.
- Do not call a generated trajectory physically valid merely because it looks visually plausible.
- Distinguish evidence from speculation.

Return two sections:
A. Evidence-supported interpretation.
B. Claims that would exceed the evidence.
```

The fixed evidence supplied with the prompt consisted of the same validation/test context table, attractor-support table, lobe-switching summary, and discretization table reported above.

## Verbatim LLM response

```text
A. Evidence-supported interpretation.

1. Context length: Increasing the tested context from m=1 to m=3 materially improved next-token prediction. Validation accuracy rose from 0.61448 at m=1 to 0.69952 at m=2 and 0.75721 at m=3. The independently reported test accuracies show the same pattern, ending at 0.75913 for the validation-selected m=3 model. Exact length-3 history coverage remains about 99.45% on test, so the improvement is not primarily produced by fallback to shorter histories. The supported conclusion is only that m=3 was best among the tested values {1,2,3}; longer contexts were not tested.

2. Attractor proximity: The primary comparison must be representation-matched because generated states are box centers. With the threshold fixed as the 95th percentile of validation box-center distances to the continuous training cloud, about 95.03% of held-out test box centers and 95.21% of generated box centers are within the threshold. Their mean nearest-neighbor distances are also similar (about 0.854 for test box centers and 0.848 for generated box centers). This supports the limited claim that generated tokens remain in roughly the same coarse phase-space support as quantized held-out Lorenz data. Because an empirical Markov model emits only training-observed next tokens, this support result is partly built in by construction; it is not independent proof of correct dynamics and does not show that generated sequences are exact Lorenz solutions.

3. Two-lobe switching: The generated sequences reproduce the tested switching statistics reasonably well. The mean switch rate is about 11.52 switches per 1000 steps for true test box-center trajectories and 11.77 for generated trajectories. Mean lobe-residence times are also close, roughly 1.67 and 1.65 time units. This is evidence for reproducing this particular coarse long-time feature, not for reproducing the full invariant measure or all chaotic statistics.

4. Discretization: The measured tradeoff is clear. With 8, 12, and 20 bins per coordinate, possible vocabulary size grows from 512 to 1,728 to 8,000; observed training vocabulary grows from 117 to 250 to 628. Test quantization RMSE improves from 3.063 to 2.042 to 1.227, but test next-token accuracy decreases from 0.7760 to 0.7591 to 0.7389 and the unseen exact length-3 history fraction rises from 0.00337 to 0.00553 to 0.02011. Finer boxes preserve more geometry but make the sequence problem larger and sparser.

5. Physical interpretation: The Lorenz system here is dissipative because the divergence is -41/3. The experiment did not use a Hamiltonian system, so energy, momentum, and Casimir conservation are not appropriate success criteria. The generated sequences should be described as samples from a learned discrete Markov model, not as numerical solutions of the Lorenz ODE.

6. Observed limitations: Prediction is not perfect even at m=3, and finer discretization reduces accuracy while increasing unseen histories. The attractor and switching tests are coarse diagnostics based on box centers and selected summary statistics. They do not test every aspect of long-horizon Lorenz dynamics.

B. Claims that would exceed the evidence.

- It would exceed the evidence to claim that m=3 is globally optimal or that more context can never help; only m=1,2,3 were tested.
- It would exceed the evidence to claim that the generated process has the same invariant measure, Lyapunov exponents, entropy, or chaotic dynamics as the Lorenz ODE; none of those quantities were measured.
- It would exceed the evidence to call the generated paths exact or physically valid Lorenz trajectories. They are discrete box-center reconstructions sampled from empirical transition probabilities.
- It would exceed the evidence to claim that finer discretization is always worse for prediction. Only three resolutions were tested with one fixed sampling interval, one data volume, and one selected context length.
- It would exceed the evidence to attribute errors specifically to insufficient training trajectories, the sampling interval, the choice of uniform boxes, or random-seed variability because those factors were not varied.
- It would be physically incorrect to discuss conservation of Hamiltonian energy, momentum, or Casimir invariants as requirements for this selected dissipative Lorenz experiment.
```

The complete machine-readable prompt and response are retained in:

- `data/hw5_2/task11_llm_critic_prompt.txt`
- `data/hw5_2/task11_llm_critic_response.txt`


# 13. Human Critique of the LLM and Interpretation Boundaries

## 13.1 What the LLM got right

The LLM response is well aligned with the measured evidence.

It correctly states that:

- $m=3$ is the strongest of the tested context lengths, not necessarily globally optimal;
- exact-history coverage at $m=3$ remains very high;
- the Task 8 comparison must use matched representations;
- generated states remain in roughly the same **coarse tokenized support** as held-out truth;
- support retention is partly built into the empirical Markov generator and is therefore weak evidence by itself;
- generated lobe-switching rate and residence-time scale are close to the held-out test summaries;
- finer discretization improves geometric resolution while increasing vocabulary size, sparsity, and prediction difficulty;
- Hamiltonian energy, momentum, and Casimir conservation are inappropriate criteria for this dissipative Lorenz experiment.

## 13.2 Claims that would exceed the evidence

The report does **not** support the stronger statements that:

- $m=3$ is the globally optimal history length;
- the generated process has the same invariant probability measure as the Lorenz flow;
- the generated process has the same Lyapunov exponents or entropy;
- a generated box-center path is an exact or physically admissible solution of the Lorenz ODE;
- finer discretization always lowers predictive accuracy;
- the remaining errors are caused specifically by too little training data, the chosen sampling interval, the uniform partition, or random-seed variability.

Those possibilities were not established by the experiment.

## 13.3 Prediction is not the same as dynamical recovery

A Markov next-token model learns empirical conditional probabilities between discrete phase-space regions.

It does **not** directly learn the continuous vector field

$$
F(x,y,z).
$$

Therefore good next-token accuracy does not imply recovery of:

- local derivatives;
- exact flow speed;
- continuous trajectories inside each box;
- Lyapunov exponents;
- invariant density;
- all long-time chaotic statistics.

## 13.4 Attractor proximity is a limited diagnostic

The nearest-neighbor support test measures whether token box centers lie near the training-state cloud.

Because the generator can sample only transitions represented in the empirical training model, coarse support retention is partly structural.

The lobe-switching comparison is more informative because it measures sequence behavior over time, but it still probes only one coarse feature of the attractor.

## 13.5 Discretization limitations

Uniform rectangular boxes are simple and reproducible, but they are not adapted to the geometry of the Lorenz attractor.

An adaptive or nonuniform partition might change the vocabulary-size and prediction tradeoff. That was not tested here.

## 13.6 Overall assessment of the LLM

The LLM is useful as an interpretive assistant because it correctly organizes the measured evidence and identifies claims that would exceed it.

Its conclusions remain subordinate to the mathematical structure of the Lorenz system and the numerical experiment. Visual plausibility, high token accuracy, or generic statements about chaos cannot replace explicit diagnostics.

The LLM response used here is appropriately conservative and does not invent Hamiltonian invariants.

# Overall Conclusions

Homework 5.2 demonstrates the central Chapter 5 idea that next-token prediction is not specific to language.

A continuous time-ordered system can be transformed into a discrete sequence problem:

$$
X(t_1),X(t_2),\ldots
\longrightarrow
a_1,a_2,\ldots.
$$

For the Lorenz experiment, three conclusions are directly supported.

## 1. Additional context helps within the tested range

Increasing the context from

$$
m=1
$$

to

$$
m=3
$$

raises final test next-token accuracy from approximately 60.7% to 75.9%, while exact length-3 history coverage remains above 99%.

## 2. Discretization creates a prediction-versus-geometry tradeoff

Coarser boxes produce a smaller vocabulary and higher prediction accuracy, while finer boxes reduce geometric quantization error but produce a larger and sparser sequence problem.

## 3. Generated sequences reproduce selected coarse Lorenz statistics

Generated sequences remain in the same coarse tokenized support as held-out truth and reproduce the tested lobe-switching summaries reasonably well.

However, the generator should be described as a learned **discrete stochastic sequence model**, not as a recovered Lorenz ODE solver.

The strongest defensible conclusion is:

> The Lorenz system can be represented as a discrete token process whose short-history statistics are sufficiently structured for useful next-token prediction and generation, but success at the token level does not by itself establish recovery of the full continuous chaotic dynamics.